# Medical Chatbot Research Notebook

This notebook demonstrates the RAG (Retrieval-Augmented Generation) pipeline for the Medical Chatbot. 
It covers data loading, text splitting, embedding generation, vector store creation (Pinecone), and the QA chain setup.

## 1. Environment Setup & Imports

Load environment variables suitable for the project and import necessary libraries.

In [3]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

c:\Users\HP 1030 G7\anaconda3\envs\iiot_env\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## 2. data Loading

We load PDF documents from the `data` directory. Note: Ensure you are in the correct directory or adjust the path.

In [4]:
# Ensure we are pointing to the correct data directory.
# If running from 'research' folder, data might be in '../data'
# os.chdir("../") # Uncomment if needed to change to root directory

import os

os.chdir("../")


def load_pdf_file(data_path):
    loader = DirectoryLoader(
        data_path,
        glob="1797022_man_EN.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

# Assuming 'data' folder is in the current or parent directory
extracted_documents = load_pdf_file("data")
print(f"Loaded {len(extracted_documents)} pages/documents.")

Loaded 180 pages/documents.


## 3. Text Splitting

Split the loaded documents into smaller chunks for efficient embedding and retrieval.

In [5]:
def text_split(docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # idéal pour diagnostic + solutions
        chunk_overlap=100,    # garde la continuité logique
        separators=[
            "\n\n",           
            "\n",             
            ".",              
            " ",              
            ""
        ]
    )
    texts_chunk = text_splitter.split_documents(docs)
    return texts_chunk

texts_chunk = text_split(extracted_documents)
print(f"Created {len(texts_chunk)} chunks.")

Created 511 chunks.


## 4. Embeddings

Initialize HuggingFace embeddings (`all-MiniLM-L6-v2`) to convert text into vector representations.

In [6]:
def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding = download_embeddings()

c:\Users\HP 1030 G7\anaconda3\envs\iiot_env\lib\site-packages\langchain_core\_api\deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


## 5. Vector Store (Pinecone)

Set up the Pinecone index and store the document embeddings. 
**Note:** Code to create a new index is included but commented out if it already exists.

In [7]:
index_name = "industrial-chatbot"
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists, create if not (Optional)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# Upload NEW documents:
# docsearch = PineconeVectorStore.from_documents(
#     documents=texts_chunk,
#     embedding=embedding,
#     index_name=index_name
# )

# Connect to the existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding,
)



## 6. Retrieval & LLM Setup

Set up the retriever and the Chat Model (using OpenRouter/OpenAI compatible interface).

In [8]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Test Retriever
retrieved_doc = retriever.invoke("Pourquoi le moteur de ma CNC ne démarre pas ?")
for doc in retrieved_doc:
    print("*"*100)
    print(doc.page_content)

****************************************************************************************************
de 2) 
1797001   Sabot pour la poussière « CNC » 
1797003   Trousse de fraises pour la fraiseuse 
« CNC »
****************************************************************************************************
13.0  Accessoires 
additionnels 
Contacter  votre fournisseur ou Powermatic pour commander. 1797000   Dispositifs de retenue « CNC » (trousse de 2) 1797001   Sabot pour la poussière « CNC » 1797003   Trousse de fraises pour la fraiseuse « CNC »
****************************************************************************************************
13.0  Accessoires 
additionnels 
Contacter  votre fournisseur ou Powermatic pour commander. 1797000   Dispositifs de retenue « CNC » (trousse de 2) 1797001   Sabot pour la poussière « CNC » 1797003   Trousse de fraises pour la fraiseuse « CNC »


In [10]:
chat = ChatOpenAI(
    model_name="openai/gpt-4o-mini", 
    openai_api_base="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

msg = [HumanMessage(content="Say hi")]
resp = chat(msg)
print(resp.content)

c:\Users\HP 1030 G7\anaconda3\envs\iiot_env\lib\site-packages\langchain_core\_api\deprecation.py:141: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
c:\Users\HP 1030 G7\anaconda3\envs\iiot_env\lib\site-packages\langchain_core\_api\deprecation.py:141: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(


Hi! How can I assist you today?


## 7. RAG Chain

Combine the retriever and the LLM into a Question-Answering chain.

In [11]:
system_prompt = """
You are an expert CNC maintenance technician. 
You MUST answer ONLY using the information explicitly present in the provided manual context.

STRICT RULES:
- Only provide the information requested in the following sections.
- Do NOT add, infer, or invent any information.
- If a section is not present in the manual, write "Information not available in the manual" for that section.
- Do NOT include any extra explanation or steps beyond the requested sections.

RESPONSE FORMAT (DO NOT CHANGE):
1. Symptom(ONLY if explicitly stated in the manual)
2. Possible causes (bullet list, ONLY if explicitly stated in the manual)
3. Diagnostic procedure (bullet list, ONLY if explicitly stated in the manual)
4. Maintenance / corrective actions (numbered steps, ONLY if explicitly stated in the manual)
5. Safety warning (ONLY if explicitly stated in the manual)
6. Manual reference (section or page if available)

MANUAL CONTEXT:
{context}
"""




prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(chat, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

## 8. Testing the Pipeline

In [ ]:
response = rag_chain.invoke({"input": "Le moteur de la machine CNC ne démarre pas et les fusibles continuent de sauter. Quelles sont les causes possibles et comment puis-je résoudre le problème ?"})
print("Answer: \n", response["answer"])

Answer: 
 1. Symptom
   - Le moteur de la machine CNC ne démarre pas et les fusibles continuent de sauter.

2. Possible causes
   - Information not available in the manual

3. Diagnostic procedure
   - Information not available in the manual

4. Maintenance / corrective actions
   - Information not available in the manual

5. Safety warning
   - Ne jamais laisser la machine en marche sans surveillance. Fermer l’alimentation électrique et ne pas quitter avant que la machine soit complètement arrêtée.

6. Manual reference
   - Information not available in the manual


In [13]:
response = rag_chain.invoke({"input": "The CNC machine motor does not start and the fuses keep blowing. What are the possible causes and how can I fix the problem?"})
print("Answer: \n", response["answer"])

Answer: 
 1. Symptom: Motor will not start: fuses blow or circuit breakers trip.
2. Possible causes:
   - Short circuit in line cord or plug.
   - Short circuit in motor or loose connections.
   - Incorrect fuses/breakers in power line.
3. Diagnostic procedure:
   - Inspect cord or plug for damaged insulation and shorted wires.
   - Inspect all connections on motor for loose or shorted terminals or worn insulation.
   - Install correct fuses or circuit breakers.
4. Maintenance / corrective actions:
   - 1. Inspect cord or plug for damaged insulation and shorted wires.
   - 2. Inspect all connections on motor for loose or shorted terminals or worn insulation.
   - 3. Install correct fuses or circuit breakers.
5. Safety warning: WARNING: Some corrections may require a qualified electrician.
6. Manual reference: 14.1 Mechanical and electrical problems


In [ ]:
response = rag_chain.invoke({"input": "The motor attempts to start but does not rotate. What diagnostic steps should I follow and what maintenance actions are recommended?"})
print("Answer:\n", response["answer"])

Answer:
 1. Symptom  
Motor attempts start, but will not turn.

2. Possible causes  
- Jammed spindle.  
- Motor faulty.  
- Spindle run without coolant.  
- Incorrect voltage.  

3. Diagnostic procedure  
- Disconnect from power, try turning spindle by hand.  
- Check reason for jamming.  
- Check incoming voltage.  

4. Maintenance / corrective actions  
1. Replace spindle (if motor is faulty).  
2. Replace motor (if spindle runs without coolant).  
3. Maintain coolant level (if spindle runs without coolant).  

5. Safety warning  
* WARNING: Some corrections may require a qualified electrician.

6. Manual reference  
Section 14.1


: 